# Reproduce unsupervised Figure 12: TEP row-index bootstrap forest

This notebook reproduces Figure 12 from the Tennessee Eastman Process (TEP) section of the paper. The PCA notebooks used the Synthetic Noise Feature (SNF) to decide which principal components and variables were more informative than noise. Here the same SNF idea is used in a supervised way: the row index is treated as the target, and a bootstrap forest ranks which process trends best explain the passage of time through the process shift.

## Reference and library

**Main article:** Vallerio, M., del Rio Chanona, A., & Navarro-Brull, F. J. (2026). *All you need is noise - from feature selection to explainable industrial AI*. **Digital Chemical Engineering, 18**, 100290. https://doi.org/10.1016/j.dche.2026.100290

**Model used:** this notebook fits a LightGBM random-forest regressor (`boosting_type="rf"`). This is the LightGBM analogue of a bootstrap forest: every tree sees a random bootstrap-like fraction of the observations and a random fraction of the variables.

**Paper idea reproduced here:** the row index is used as an independent monotone target. Process variables that change strongly and systematically over the plant shift can predict row number better than the random-noise tag. The `Random Normal` SNF is therefore used as the visual and numerical noise floor for the ranked variables.

In [ ]:
# Self-contained runtime setup for reproducing the paper figure.
import importlib.util
import subprocess
import sys


def ensure(import_name, package_name=None):
    package_name = package_name or import_name
    if importlib.util.find_spec(import_name) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", package_name])


for import_name, package_name in [
    ("numpy", "numpy"),
    ("pandas", "pandas"),
    ("openpyxl", "openpyxl"),
    ("matplotlib", "matplotlib"),
    ("seaborn", "seaborn"),
    ("sklearn", "scikit-learn"),
    ("lightgbm", "lightgbm"),
]:
    ensure(import_name, package_name)

# On macOS, LightGBM may also need the OpenMP runtime.
# If importing LightGBM fails with a libomp error, install it once with:
# conda install -c conda-forge llvm-openmp

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from lightgbm import LGBMRegressor

warnings.filterwarnings("ignore", category=UserWarning)

plt.rcParams.update({
    "figure.dpi": 130,
    "savefig.dpi": 220,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.22,
    "font.size": 10,
})
sns.set_theme(style="whitegrid", context="notebook", font_scale=0.92)


PALETTE = {
    "target": "#7aa6ff",
    "important": "#222222",
    "less_important": "#c7c7c7",
    "noise": "#d62728",
    "bar": "#4d4d4d",
}


def find_unsupervised_root():
    """Locate the 02_unsupervised folder from this notebook or current working directory."""
    candidates = [Path.cwd(), *Path.cwd().parents]
    fallback = Path("/Users/b42549592/Documents/GitHub/all-you-need-is-noise/02_unsupervised")
    candidates.extend([fallback, *fallback.parents])
    for candidate in candidates:
        if candidate.name == "02_unsupervised" and (candidate / "01_density_dataset").exists() and (candidate / "02_TEP").exists():
            return candidate
        nested = candidate / "02_unsupervised"
        if (nested / "01_density_dataset").exists() and (nested / "02_TEP").exists():
            return nested
    raise FileNotFoundError("Could not locate the 02_unsupervised folder")


UNSUPERVISED_ROOT = find_unsupervised_root()
TEP_ROOT = UNSUPERVISED_ROOT / "02_TEP"
OUTPUT_DIR = TEP_ROOT / "results" / "python" / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
DATA_PATH = TEP_ROOT / "Tennesse MDMSPC - PCA - Eastman Chemical.xlsx"
raw = pd.read_excel(DATA_PATH)

# The PCA reproduction used JMP's `measurement vars` column group.
# Figure 12 uses the same process measurements plus the random-noise tag.
measurement_cols = list(raw.columns[4:26])
feature_cols = measurement_cols + ["Random Normal"]
target_col = "Row number"

X = raw[feature_cols].apply(pd.to_numeric, errors="coerce")
y = raw[target_col].astype(float)

print(f"Loaded {len(raw)} TEP observations")
print(f"Model inputs: {len(feature_cols)} variables, including the Random Normal SNF")
display(raw[["Datetime", target_col, "type"] + feature_cols[:5]].head())

## Fit the bootstrap-forest model

The row number is not a process measurement. It is used only as a monotone reference for time. If a sensor trend changes with the plant shift, a supervised model can use that sensor to predict the row index. Variables that cannot predict row index better than the random tag are treated as low-information for this anomaly screen.

In [ ]:
# LightGBM's random forest mode needs bagging on rows and variables.
# These settings keep the model simple and close to the paper's bootstrap-forest idea.
forest = LGBMRegressor(
    boosting_type="rf",
    n_estimators=300,
    bagging_fraction=0.632,
    bagging_freq=1,
    feature_fraction=0.8,
    objective="regression",
    random_state=42,
    n_jobs=-1,
    verbosity=-1,
)
forest.fit(X, y)

# Use split gain because it measures how much each variable helped the trees.
# Convert the gains to fractions so the bars add up to 1.0.
gain = forest.booster_.feature_importance(importance_type="gain")
importance = pd.DataFrame({
    "variable": feature_cols,
    "gain": gain,
})
importance["contribution_fraction"] = importance["gain"] / importance["gain"].sum()
importance["contribution_percent"] = 100 * importance["contribution_fraction"]
importance = importance.sort_values("contribution_fraction", ascending=False).reset_index(drop=True)

noise_contribution = importance.loc[
    importance["variable"].eq("Random Normal"), "contribution_fraction"
].iloc[0]
importance["above_snf"] = importance["contribution_fraction"] > noise_contribution

importance_path = OUTPUT_DIR / "figure_12_lightgbm_bootstrap_forest_contributions.csv"
importance.to_csv(importance_path, index=False)

print(f"Random Normal contribution: {noise_contribution:.4%}")
print(f"Saved contribution table to: {importance_path}")
display(importance.round({"gain": 2, "contribution_fraction": 5, "contribution_percent": 3}).head(25))

## Figure 12: row-index target and ranked process trends

**Article explanation:** Figure 12 turns the TEP process shift into a supervised anomaly screen. The model uses the full process-trend table to predict row index. The resulting feature importance ranks the variables by how strongly their trends explain the process shift. The SNF is plotted in red and acts as the cutoff between variables with stronger-than-noise trend information and variables near the noise floor.

In [ ]:
# Order the trends directly from the LightGBM contribution ranking.
ranked_vars = importance["variable"].tolist()
plot_order = [target_col] + ranked_vars
n_lanes = len(plot_order)

# Scale every displayed trace to 0-1 before stacking it in a lane.
# This keeps flow, pressure, temperature, and composition variables readable together.
trend_data = pd.concat([raw[["Datetime", target_col]], X[ranked_vars]], axis=1).copy()
scaled = trend_data[plot_order].apply(pd.to_numeric, errors="coerce")
scaled = (scaled - scaled.min()) / (scaled.max() - scaled.min())
scaled = scaled.fillna(0.5)

fig = plt.figure(figsize=(14.4, 8.2), constrained_layout=True)
gs = fig.add_gridspec(1, 2, width_ratios=[7.8, 1.1], wspace=0.035)
ax_trend = fig.add_subplot(gs[0, 0])
ax_bar = fig.add_subplot(gs[0, 1], sharey=ax_trend)

lane_positions = np.arange(n_lanes)[::-1]
lane_lookup = dict(zip(plot_order, lane_positions))

for variable in plot_order:
    y0 = lane_lookup[variable]
    if variable == target_col:
        color = PALETTE["target"]
        lw = 1.35
        alpha = 1.0
    elif variable == "Random Normal":
        color = PALETTE["noise"]
        lw = 1.0
        alpha = 1.0
    else:
        is_above_noise = bool(importance.loc[importance["variable"].eq(variable), "above_snf"].iloc[0])
        color = PALETTE["important"] if is_above_noise else PALETTE["less_important"]
        lw = 0.95
        alpha = 0.94 if is_above_noise else 0.70
    ax_trend.plot(trend_data["Datetime"], scaled[variable] * 0.72 + y0 - 0.36,
                  color=color, lw=lw, alpha=alpha)

# Draw a light separator between trend lanes.
for y0 in lane_positions:
    ax_trend.axhline(y0 - 0.47, color="#eeeeee", lw=0.5, zorder=0)

ax_trend.set_yticks(lane_positions)
ax_trend.set_yticklabels(plot_order, fontsize=8)
ax_trend.set_ylim(-0.8, n_lanes - 0.2)
ax_trend.set_xlabel("Datetime")
ax_trend.set_ylabel("")
ax_trend.set_title("(a) Row-index target and process trends ordered by model contribution", loc="left", fontweight="bold")
ax_trend.grid(axis="x", alpha=0.25)
ax_trend.grid(axis="y", alpha=0.04)

# Keep dates readable without adding extra formatting complexity.
for label in ax_trend.get_xticklabels():
    label.set_rotation(0)
    label.set_horizontalalignment("center")

# Feature-importance bars use the same y positions as the trend panel.
bar_y = [lane_lookup[v] for v in ranked_vars]
bar_colors = [PALETTE["noise"] if v == "Random Normal" else PALETTE["bar"] for v in ranked_vars]
ax_bar.barh(bar_y, importance["contribution_fraction"], height=0.62, color=bar_colors)
ax_bar.axvline(noise_contribution, color=PALETTE["noise"], lw=1.3, linestyle=":")
ax_bar.set_xlabel("Contribution fraction")
ax_bar.set_title("(b) Contribution\nLightGBM bootstrap forest", loc="left", fontweight="bold", fontsize=9)
ax_bar.tick_params(axis="y", left=False, labelleft=False)
ax_bar.grid(axis="x", alpha=0.25)
ax_bar.grid(axis="y", alpha=0.0)
ax_bar.set_xlim(0, max(importance["contribution_fraction"].max() * 1.10, noise_contribution * 4))

# Highlight the SNF label in red in the trend panel.
for tick in ax_trend.get_yticklabels():
    if tick.get_text() == "Random Normal":
        tick.set_color(PALETTE["noise"])
        tick.set_fontweight("bold")
    elif tick.get_text() == target_col:
        tick.set_color(PALETTE["target"])
        tick.set_fontweight("bold")

fig.suptitle("Figure 12 reproduction: supervised anomaly screen with row index as target", fontweight="bold")
figure_path = OUTPUT_DIR / "figure_12_tep_lightgbm_bootstrap_forest.png"
fig.savefig(figure_path, bbox_inches="tight")
print(f"Saved figure to: {figure_path}")
plt.show()

**Figure 12 caption.** Demonstration of the supervised SNF anomaly screen on the Tennessee Eastman dataset. Panel (a) shows the row-index target and the TEP process trends ordered by their LightGBM bootstrap-forest contribution. Panel (b) shows the normalized feature-importance contributions, with `Random Normal` in red marking the stronger-than-noise threshold for informative process-shift variables.